# 01 · 데이터 다운로드 (D1a, 구현서 §5.1~5.2)

- **무엇을:** `data/sources.yaml`의 소스 12개를 각자 컨테이너 하나씩 써서 병렬로 받습니다. 소스별 목표량은 (read_gb + l3_gb) × 1.6이고, 합계 약 **99GB의 텍스트**를 Modal Volume `askos-data`의 `/raw/<소스>/`에 `*.jsonl.zst`로 저장합니다.
- **얼마나:** 소스마다 몇십 분에서 몇 시간이 걸립니다. 가장 긴 것은 FineWeb-Edu, DCLM, Stack-Edu입니다. GPU는 쓰지 않고 CPU 4코어·RAM 32GB 컨테이너 12개를 씁니다. 실행 시간은 실측 전이라 아직 모릅니다.
- **결과:** `reports/D1.md`를 만듭니다. 소스별 실제 바이트, 데이터셋 리비전, verify 결과, 체크섬, 약관 메모가 들어가고, Volume과 HF 비공개 저장소 `reports/D1.md`에 같이 남습니다.

> 끊겨도 됩니다. 각 소스는 `_state.json`에서 이어 받고, 다 받은 소스는 다시 실행해도 바로 끝납니다.

In [ ]:
!pip install -q "modal>=1.0" pyyaml numpy

## 1. 로컬 시험 (GPU·Modal·네트워크 없이)

In [ ]:
!python3 tests/test_infra.py && python3 tests/test_download.py && python3 -m py_compile modal_app.py ops.py data/*.py data/readers/*.py && echo "로컬 시험 통과"

## 2. 목표량 확인 (네트워크 없음)

In [ ]:
!PYTHONPATH=. python3 -m data.download plan

## 3. 실행 (한 번만)

In [ ]:
!bash launch.sh download

## 4. 상태 확인

15~20분마다 다시 실행합니다. `RUNNING`이면 소스별 진행률 표가 같이 나옵니다(Volume에 5분마다 반영).

In [ ]:
!bash status.sh download

## 5. 결과

In [ ]:
import json, pathlib
out = pathlib.Path("runs/01_download")
s = json.loads((out / "summary.json").read_text())
print("판정:", s["verdict"])
print((out / "D1.md").read_text()[:6000])

## 다음

- **통과**면 `02_data_clean_dedup_split.ipynb`로 넘어갑니다.
- **미완**이면 D1.md에서 `실패`, `미달`인 소스와 `오류` 줄을 보고, `AGENT_INSTRUCTIONS.md`의 01 문제 대처표대로 처리합니다. 사람이 승인하면 `bash launch.sh download --again`으로 다시 실행합니다. 이미 받은 부분은 다시 받지 않습니다.